# Aula 3 · Lab 4 — Replicação e falhas de broker

**Objetivo:** derrubar brokers de verdade e observar eleição de líder, ISR e o efeito de
`acks=all` + `min.insync.replicas=2`.

Os comandos que derrubam brokers rodam **no Azure Cloud Shell** (de dentro da pasta do repositório),
porque o Jupyter não controla os containers:
```bash
bash infra/vm.sh "docker stop kafka2"
bash infra/vm.sh "docker start kafka2"
```

In [ ]:
import json, time, threading
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
from comum.config import BOOTSTRAP

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
TOPICO = "aula3-replicado"
try:
    admin.create_topics([NewTopic(TOPICO, 3, 3, config={"min.insync.replicas": "2"})])[TOPICO].result()
except Exception as e:
    print("aviso:", e)

def descrever():
    meta = admin.list_topics(topic=TOPICO, timeout=10)
    vivos = sorted(meta.brokers)
    print(time.strftime("%H:%M:%S"), "brokers vivos:", vivos)
    for p in sorted(meta.topics[TOPICO].partitions.values(), key=lambda p: p.id):
        print(f"  partição {p.id}: líder={p.leader} réplicas={p.replicas} ISR={p.isrs}")

descrever()

## 1. Um produtor contínuo
Esta célula inicia um produtor em segundo plano que envia 5 mensagens/s com `acks=all`
e conta sucessos e erros.

In [ ]:
estatisticas = {"ok": 0, "erro": 0, "ultimo_erro": None}
rodando = threading.Event()
rodando.set()

def produzir():
    p = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True,
                  "delivery.timeout.ms": 10000, "request.timeout.ms": 5000})
    def cb(err, msg):
        if err:
            estatisticas["erro"] += 1
            estatisticas["ultimo_erro"] = str(err)
        else:
            estatisticas["ok"] += 1
    i = 0
    while rodando.is_set():
        p.produce(TOPICO, key=f"k{i % 9}", value=json.dumps({"i": i}), callback=cb)
        p.poll(0.2)
        i += 1
    p.flush(15)

threading.Thread(target=produzir, daemon=True).start()
time.sleep(3)
print(estatisticas)

## 2. Derrube um broker
No **Cloud Shell**: `bash infra/vm.sh "docker stop kafka2"`

Espere ~10 segundos e rode a célula abaixo algumas vezes.

- As partições que tinham o broker 2 como líder ganharam um novo líder?
- O broker 2 saiu do ISR?
- O produtor continuou com sucesso? (ISR com 2 réplicas ≥ `min.insync.replicas=2`)

In [ ]:
descrever()
print(estatisticas)

## 3. Derrube um segundo broker
No Cloud Shell: `bash infra/vm.sh "docker stop kafka3"`

Agora só sobra 1 réplica. Com `acks=all` e `min.insync.replicas=2`, o que acontece com as escritas?
Rode a célula abaixo depois de ~15 segundos.

> Com 2 de 3 controladores fora, o quórum do KRaft também se perde: os metadados podem ficar desatualizados
> ou indisponíveis. Isso faz parte da lição.

In [ ]:
try:
    descrever()
except Exception as e:
    print("não consegui ler metadados:", e)
print(estatisticas)

## 4. Recuperação
No Cloud Shell: `bash infra/vm.sh "docker start kafka2 kafka3"`

Rode a célula abaixo depois de ~20 segundos. As réplicas voltam para o ISR?
A liderança volta a ficar equilibrada? (o Kafka faz *preferred leader election* periodicamente)

In [ ]:
descrever()
print(estatisticas)

In [ ]:
rodando.clear()   # para o produtor de fundo
time.sleep(2)
print("final:", estatisticas)

## Para discutir
1. Com `acks=1`, o que poderia ser **perdido** se o líder caísse logo após confirmar a escrita?
2. Por que `replication_factor=3` + `min.insync.replicas=2` é a configuração mais comum em produção?
3. Qual é a diferença entre um broker fora do ar e um broker fora do ISR?
4. Aqui os 3 brokers estão na **mesma VM**. O que isso significa para a tolerância a falhas de verdade?
   (Ligue com *rack awareness* e zonas de disponibilidade.)